<a href="https://colab.research.google.com/github/Zahra-ah/object_detection/blob/main/ObjectDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit
!pip install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 39.3 MB/s eta 0:00:00


In [2]:
!pip install -q mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.8 MB/s eta 0:00:00


In [3]:
!wget -q -O efficientdet.tflite -q https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/int8/1/efficientdet_lite0.tflite

In [4]:
%%writefile ObjectDetection.py
import streamlit as st
from PIL import Image
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
st.title("Object Detection Application")
st.text("Built with Streamlit and OpenCV")
uploaded_img=st.file_uploader("upload image",type=['jpg','png','jpeg'])
import cv2
import numpy as np

MARGIN = 10  # pixels
ROW_SIZE = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
TEXT_COLOR = (255, 0, 0)  # red


def visualize(
    image,
    detection_result
) -> np.ndarray:
  """Draws bounding boxes on the input image and return it.
  Args:
    image: The input RGB image.
    detection_result: The list of all "Detection" entities to be visualize.
  Returns:
    Image with bounding boxes.
  """
  for detection in detection_result.detections:
    # Draw bounding_box
    bbox = detection.bounding_box
    start_point = bbox.origin_x, bbox.origin_y
    end_point = bbox.origin_x + bbox.width, bbox.origin_y + bbox.height
    cv2.rectangle(image, start_point, end_point, TEXT_COLOR, 3)

    # Draw label and score
    category = detection.categories[0]
    category_name = category.category_name
    probability = round(category.score, 2)
    result_text = category_name + ' (' + str(probability) + ')'
    text_location = (MARGIN + bbox.origin_x,
                     MARGIN + ROW_SIZE + bbox.origin_y)
    cv2.putText(image, result_text, text_location, cv2.FONT_HERSHEY_PLAIN,
                FONT_SIZE, TEXT_COLOR, FONT_THICKNESS)

  return image
# STEP 2: Create an ObjectDetector object.
base_options = python.BaseOptions(model_asset_path='efficientdet.tflite')
options = vision.ObjectDetectorOptions(base_options=base_options,
                                       score_threshold=0.5)
detector = vision.ObjectDetector.create_from_options(options)
if uploaded_img is not None:
  file_bytes=np.asarray(bytearray(uploaded_img.read()),dtype=np.uint8)
  img=cv2.imdecode(file_bytes,1)
  img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

  st.image(img,caption='Uploaded Image',use_column_width=True)
  mp_image=mp.Image(image_format=mp.ImageFormat.SRGB,data=img)
  detection_result = detector.detect(mp_image)
  annotated_image = visualize(img, detection_result)
  st.image(annotated_image,use_column_width=True)


Writing ObjectDetection.py


In [5]:
from pyngrok import ngrok
!ngrok authtoken ----

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [6]:
!streamlit run ObjectDetection.py &>/dev/null&

In [7]:
!pgrep -f streamlit

830


In [12]:
ngrok.kill()

In [13]:
puplic_url=ngrok.disconnect(8501)
puplic_url